# Ejercicio: Web Scraping

## Objetivo de la práctica

El objetivo de este ejercicio es construir un web scraper que recoja datos de un website.

### Parte 0: Planificar
1. Identificar los datos que quieres obtener.
2. Elegir el sitio web objetivo.
3. Planificar la estructura del corpus.

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos bsuscados.

In [1]:
from bs4 import BeautifulSoup

file = '../data/rotisserie-chicken.html'

# Load the HTML file
with open(file, "r", encoding="utf-8") as file:
    html_content = file.read()
    
# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

In [2]:
# Extracting the recipe title
title = soup.find("meta", {"property": "og:title"})["content"]
title

'Rotisserie Chicken'

In [3]:
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
for ingredient in ingredients_section:
    print(ingredient.text.strip())

1 (3 pound) whole chicken
1 pinch salt
¼ cup butter, melted
1 tablespoon salt
1 tablespoon ground paprika
¼ tablespoon ground black pepper


## Parte 2: Obtener los datos deseados

* Buscar dentro del contenido HTML y extraer la información.

In [4]:
# Extracting the description
description = soup.find("meta", {"name": "description"})["content"]

# Extracting the ingredients
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]

# Extracting the instructions
instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
instructions = [instruction.get_text().strip() for instruction in instructions_section]

# Extracting the nutrition information
nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]

# Print the extracted information
print("Recipe Title:", title)
print("Description:", description)
print("Ingredients:")
for ingredient in ingredients:
    print("-", ingredient)
print("Instructions:")
for i, instruction in enumerate(instructions, 1):
    print(f"{i}. {instruction}")
print("Nutrition Facts:")
for fact in nutrition_facts:
    print("-", fact)


Recipe Title: Rotisserie Chicken
Description: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.
Ingredients:
- 1 (3 pound) whole chicken
- 1 pinch salt
- ¼ cup butter, melted
- 1 tablespoon salt
- 1 tablespoon ground paprika
- ¼ tablespoon ground black pepper
Instructions:
1. Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.
2. Here's what you'll need to make rotisserie chicken at home:
3. · Whole Chicken: This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time.· Butter: Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to.· Seasonings: The rotisserie chicken is simply seasoned with salt, pepper, and paprika.
4. You'll find the full, step-by-step recipe below — b

In [5]:
def extraer_datos_receta(html_content):
    """
    Recibe el HTML de una página de Allrecipes, extrae la información clave
    y la devuelve en un diccionario estructurado.
    """
    soup_receta = BeautifulSoup(html_content, "html.parser")
    
    try:
        # 1. Título de la receta
        title_tag = soup_receta.find("meta", {"property": "og:title"})
        title = title_tag["content"] if title_tag else "Sin título"
        
        # 2. Descripción
        desc_tag = soup_receta.find("meta", {"name": "description"})
        description = desc_tag["content"] if desc_tag else "Sin descripción"
        
        # 3. Ingredientes
        ingredients_section = soup_receta.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
        ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]
        
        # 4. Instrucciones
        instructions_section = soup_receta.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
        instructions = [instruction.get_text().strip() for instruction in instructions_section]
        
        # 5. Información Nutricional
        nutrition_section = soup_receta.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
        nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]
        
        # Retornamos los datos estructurados
        return {
            "titulo": title,
            "descripcion": description,
            "ingredientes": ingredients,
            "instrucciones": instructions,
            "nutricion": nutrition_facts
        }
        
    except Exception as e:
        print(f"Error al parsear los elementos de la receta: {e}")
        return None

## Parte 3: Obtener enlaces relacionados
* Encontrar links a otras recetas para completar el corpus

In [6]:
# Filtrar y limpiar los enlaces para quedarnos solo con recetas individuales
recipe_urls = []

for link in soup.find_all("a", href=True):
    href = link['href']
    
    # Condición estricta: debe contener la estructura exacta de una receta individual
    if "allrecipes.com/recipe/" in href:
        # Evitamos duplicados en nuestra lista
        if href not in recipe_urls:
            recipe_urls.append(href)

# Ver cuantas recetas reales encontramos y mostrar las primeras 5
print(f"Total de recetas individuales encontradas: {len(recipe_urls)}")
for url in recipe_urls[:16]:
    print(url)

Total de recetas individuales encontradas: 16
https://www.allrecipes.com/recipe/238575/cilantro-lime-grilled-chicken/
https://www.allrecipes.com/recipe/275062/buttermilk-barbecue-chicken/
https://www.allrecipes.com/recipe/274724/grilled-spatchcocked-chicken/
https://www.allrecipes.com/recipe/14531/beer-butt-chicken/
https://www.allrecipes.com/recipe/221093/good-frickin-paprika-chicken/
https://www.allrecipes.com/recipe/264278/miso-honey-chicken/
https://www.allrecipes.com/recipe/258659/rosemary-buttermilk-chicken/
https://www.allrecipes.com/recipe/222936/smoked-beer-butt-chicken/
https://www.allrecipes.com/recipe/228070/the-best-beer-can-chicken-ever/
https://www.allrecipes.com/recipe/214619/bbq-beer-can-chicken/
https://www.allrecipes.com/recipe/19944/drunk-chicken/
https://www.allrecipes.com/recipe/275044/grilled-chicken-under-a-brick/
https://www.allrecipes.com/recipe/281255/smoked-whole-chicken/
https://www.allrecipes.com/recipe/34957/easy-barbeque-chicken/
https://www.allrecipes.c

In [7]:
import requests
import time
import random  # <-- Librería para generar el comportamiento aleatorio humano
from bs4 import BeautifulSoup

# Aquí guardaremos todas las recetas extraídas
corpus_recetas = []

# Cabeceras completas que simulan un navegador real
headers_humanos = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
    "Referer": "https://www.google.com/"  # Le hacemos creer que venimos desde una búsqueda de Google
}

print("Iniciando la extracción con simulación de comportamiento humano...\n")

for i, url in enumerate(recipe_urls, 1):
    print(f"[{i}/{len(recipe_urls)}] Navegando a: {url}")
    
    try:
        # Realizamos la petición usando nuestras cabeceras simuladas
        respuesta = requests.get(url, headers=headers_humanos, timeout=10)
        
        if respuesta.status_code == 200:
            # Parseamos el contenido HTML directamente aquí con BeautifulSoup
            html_content = respuesta.text
            soup_receta = BeautifulSoup(html_content, "html.parser")
            
            # Usamos tu función para extraer los datos pasándole el HTML limpio
            datos_receta = extraer_datos_receta(html_content)
            
            if datos_receta:
                datos_receta["url"] = url
                corpus_recetas.append(datos_receta)
                print(f" -> ¡Éxito! Receta guardada: {datos_receta['titulo']}")
        else:
            print(f" -> Error {respuesta.status_code}: Bloqueado por el servidor.")
            
    except Exception as e:
        print(f" -> Error de conexión: {e}")
        
    # --- SLEEP HUMANO ---
    # Genera un tiempo de espera aleatorio con decimales (ej. 4.32 segundos o 6.15 segundos)
    tiempo_espera = random.uniform(3.5, 7.2)
    
    # No hacemos pausa en la última receta
    if i < len(recipe_urls):
        print(f" Esperando como un humano por {tiempo_espera:.2f} segundos antes de la siguiente receta...\n")
        time.sleep(tiempo_espera)

print("\n--- Proceso Finalizado ---")
print(f"Total de recetas recopiladas con éxito en el corpus: {len(corpus_recetas)}")

Iniciando la extracción con simulación de comportamiento humano...

[1/16] Navegando a: https://www.allrecipes.com/recipe/238575/cilantro-lime-grilled-chicken/
 -> ¡Éxito! Receta guardada: Cilantro-Lime Grilled Chicken
 Esperando como un humano por 6.87 segundos antes de la siguiente receta...

[2/16] Navegando a: https://www.allrecipes.com/recipe/275062/buttermilk-barbecue-chicken/
 -> ¡Éxito! Receta guardada: Buttermilk Barbecue Chicken
 Esperando como un humano por 3.59 segundos antes de la siguiente receta...

[3/16] Navegando a: https://www.allrecipes.com/recipe/274724/grilled-spatchcocked-chicken/
 -> ¡Éxito! Receta guardada: Grilled Spatchcocked Chicken
 Esperando como un humano por 3.89 segundos antes de la siguiente receta...

[4/16] Navegando a: https://www.allrecipes.com/recipe/14531/beer-butt-chicken/
 -> ¡Éxito! Receta guardada: Beer Butt Chicken
 Esperando como un humano por 4.20 segundos antes de la siguiente receta...

[5/16] Navegando a: https://www.allrecipes.com/reci

## Parte 4: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

In [8]:
# Lista donde guardaremos los documentos formateados para el RAG
documentos_rag = []

print("Preparando los textos para el sistema RAG...\n")

for receta in corpus_recetas:
    # Juntamos los ingredientes en un solo texto separado por comas o viñetas
    ingredientes_texto = ", ".join(receta["ingredientes"]) if receta["ingredientes"] else "No especificados"
    
    # Juntamos las instrucciones numeradas
    instrucciones_texto = " ".join([f"{i+1}. {inst}" for i, inst in enumerate(receta["instrucciones"])]) if receta["instrucciones"] else "No especificadas"
    
    # Juntamos los datos nutricionales
    nutricion_texto = ", ".join(receta["nutricion"]) if receta["nutricion"] else "No especificada"
    
    # Creamos un bloque de texto consolidado por receta
    texto_completo = (
        f"Receta: {receta['titulo']}\n"
        f"Descripción: {receta['descripcion']}\n"
        f"Ingredientes: {ingredientes_texto}\n"
        f"Instrucciones: {instrucciones_texto}\n"
        f"Información Nutricional: {nutricion_texto}\n"
        f"Enlace de origen: {receta['url']}"
    )
    
    # Guardamos el texto estructurado junto a sus metadatos
    documentos_rag.append({
        "text": texto_completo,
        "metadata": {
            "titulo": receta["titulo"],
            "url": receta["url"]
        }
    })

print(f"¡Listo! Se han preparado {len(documentos_rag)} documentos para el RAG.")
print("\nEjemplo del primer documento estructurado:\n")
print(documentos_rag[0]["text"][:400] + "...") # Mostramos solo el inicio para verificar

Preparando los textos para el sistema RAG...

¡Listo! Se han preparado 16 documentos para el RAG.

Ejemplo del primer documento estructurado:

Receta: Cilantro-Lime Grilled Chicken
Descripción: This cilantro-lime grilled chicken recipe starts with a quick four-ingredient marinade with lime juice, cilantro, garlic salt, and black pepper.
Ingredientes: ½ cup chopped fresh cilantro, 4  limes, juiced, 2 tablespoons garlic salt, 2 tablespoons ground black pepper, 1 whole whole chicken, cut into 6 pieces
Instrucciones: 1. Whisk cilantro, lime ...


In [ ]:
from google import genai
from google.genai import types

# Configura tu API Key de Google AI Studio
# Reemplaza 'TU_API_KEY_AQUÍ' con tu clave real
import os
os.environ["GEMINI_API_KEY"] = "your actual_api_key_here"  # <-- Asegúrate de reemplazar esto con tu clave real

# Inicializamos el cliente oficial
client = genai.Client()
print("Cliente de Gemini configurado correctamente.")

Cliente de Gemini configurado correctamente.


In [18]:
def consultar_chef_rag(ingredientes_nevera):
    # 1. Convertimos todo nuestro corpus preparado en un solo gran bloque de texto limpio
    contexto_recetas = "\n\n===\n\n".join([doc["text"] for doc in documentos_rag])
    
    # 2. Diseñamos el prompt del sistema y el contexto que alimentará al LLM
    prompt_sistema = (
        "Eres un chef experto y un asistente de cocina inteligente. Tu tarea es recomendar recetas "
        "basándote ÚNICAMENTE en el corpus de recetas provisto a continuación. No inventes recetas "
        "fuera de este contexto.\n\n"
        f"--- INICIO DEL CORPUS DE RECETAS ---\n{contexto_recetas}\n--- FIN DEL CORPUS DE RECETAS ---\n"
    )
    
    prompt_usuario = (
        f"Tengo los siguientes ingredientes en mi nevera: {ingredientes_nevera}.\n"
        "¿Qué recetas de mi corpus me recomiendas preparar? Justifica brevemente por qué "
        "e indica qué ingredientes me faltarían (si aplica). Por favor, incluye al final de cada recomendación "
        "el 'Enlace de origen' tal como aparece en el documento."
    )
    
    print("Enviando contexto y consulta a Gemini...")
    
    # 3. Llamamos al modelo a través de la API
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt_usuario,
        config=types.GenerateContentConfig(
            system_instruction=prompt_sistema,
            temperature=0.3 # Temperatura baja para que sea preciso con los datos del corpus
        )
    )
    
    return response.text

In [19]:
mis_ingredientes = "Pollo entero, mantequilla, sal, paprika, pimienta negra"

# Ejecutamos la consulta
resultado = consultar_chef_rag(mis_ingredientes)

print("\n--- RECOMENDACIÓN DEL CHEF GEMINI (RAG) ---")
print(resultado)

Enviando contexto y consulta a Gemini...

--- RECOMENDACIÓN DEL CHEF GEMINI (RAG) ---
¡Claro que sí, chef! Basándome en los ingredientes que tienes y en nuestro corpus de recetas, te recomiendo la siguiente opción:

### Receta recomendada:

**Beer Butt Chicken**

*   **Justificación:** Esta receta es una excelente opción porque utiliza todos los ingredientes que tienes: **pollo entero, mantequilla, sal, paprika y pimienta negra**. Es una forma deliciosa y popular de cocinar pollo a la parrilla, resultando en una carne muy jugosa y llena de sabor.
*   **Ingredientes que te faltarían:** Solo necesitarías **1 lata de cerveza** (de 12 onzas líquidas).
*   **Enlace de origen:** https://www.allrecipes.com/recipe/14531/beer-butt-chicken/

¡Espero que disfrutes preparando esta receta!
